# Multi-threading vs Parallel Processing in Python
**Lecture 14: Concurrency vs. Parallelism, Threading, Multiprocessing & The Python GIL**  
*Last Updated: 23 Oct, 2026*

---

### Introduction: Efficiency in Python
**Multi-threading** and **parallel processing** are two techniques used to execute multiple tasks more efficiently in Python. They sound similar, but they work differently.

Let's understand them with real-world examples and Python code.


---
## 1. What is Multi-threading?

**Multi-threading** means running multiple threads within the same process.

A **thread** is a smaller unit of execution inside a process. Threads share the same memory and resources of their parent process.

### Real-World Example
Imagine you are working in a restaurant:
- **Thread 1:** Takes an order.
- **Thread 2:** Prepares a bill.
- **Thread 3:** Answers a phone call.

While one task is waiting, another can make progress. This is particularly useful for tasks involving waiting, such as:
- API calls
- Downloading files
- Reading files
- Database queries


In [1]:
import threading
import time

def task(name):
    print(f"{name} started")
    time.sleep(2)  # Simulates waiting for an API response
    print(f"{name} completed")

start = time.time()

t1 = threading.Thread(target=task, args=("Task A",))
t2 = threading.Thread(target=task, args=("Task B",))
t3 = threading.Thread(target=task, args=("Task C",))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()

print(f"Total time: {time.time() - start:.2f} seconds")


Task A started
Task B started
Task C started
Task A completed
Task B completed
Task C completed
Total time: 2.00 seconds


### Expected Output (order may vary):
```text
Task A started
Task B started
Task C started
Task A completed
Task B completed
Task C completed
Total time: 2.00 seconds
```

### Why does this take approximately 2 seconds rather than 6 seconds?
All three threads can wait at the same time.

### Multi-threading Timeline
Each block represents a 2-second wait:

```text
Sequential Timeline (Total: ~6.0s):
[ Task A: 2s Wait ] ───────> [ Task B: 2s Wait ] ───────> [ Task C: 2s Wait ]
|___________________|______________________|______________________|
0s                  2s                     4s                     6s

Multi-Threading Timeline (Total: ~2.0s):
Task A: [====== 2.0s Waiting / I/O ======] -> Completed
Task B: [====== 2.0s Waiting / I/O ======] -> Completed
Task C: [====== 2.0s Waiting / I/O ======] -> Completed
        +────────────────────────────────+
        0s                              2.0s
```

- **Task A:** Waiting / I/O
- **Task B:** Waiting / I/O
- **Task C:** Waiting / I/O

**All three waits overlap: approximately 2 seconds total.**

> **Important:** In standard CPython builds, the **Global Interpreter Lock (GIL)** generally prevents multiple threads from executing Python bytecode simultaneously. However, threads can still overlap I/O operations effectively because CPython releases the GIL during blocking I/O (like `time.sleep`, file access, or socket communication).


---
## 2. What is Parallel Processing?

**Parallel processing** means executing multiple tasks simultaneously using multiple processes, often on different CPU cores.

Each process has its own Python interpreter and separate memory space.

### Real-World Example
Imagine four workers, each with their own workstation:
- **Worker 1:** Calculates numbers in Task A.
- **Worker 2:** Calculates numbers in Task B.
- **Worker 3:** Calculates numbers in Task C.
- **Worker 4:** Calculates numbers in Task D.

All four workers can perform CPU-intensive calculations at the same time, provided sufficient CPU cores and resources are available.


In [2]:
import multiprocessing
import os
import time

def addition(a, b):
    print(f"Addition running | PID: {os.getpid()}")
    result = a + b
    print(f"Addition result: {result}")

def subtraction(a, b):
    print(f"Subtraction running | PID: {os.getpid()}")
    result = a - b
    print(f"Subtraction result: {result}")

def multiplication(a, b):
    print(f"Multiplication running | PID: {os.getpid()}")
    result = a * b
    print(f"Multiplication result: {result}")

if __name__ == "__main__":
    # Sample inputs (or input from user: float(input(...)))
    num1 = 10.0
    num2 = 5.0
    print(f"Inputs: num1 = {num1}, num2 = {num2}\n")

    start = time.time()

    # Create separate processes
    p1 = multiprocessing.Process(
        target=addition, args=(num1, num2)
    )

    p2 = multiprocessing.Process(
        target=subtraction, args=(num1, num2)
    )

    p3 = multiprocessing.Process(
        target=multiplication, args=(num1, num2)
    )

    # Start all processes
    p1.start()
    p2.start()
    p3.start()

    # Wait for all processes to finish
    p1.join()
    p2.join()
    p3.join()

    print(f"\nTotal execution time: {time.time() - start:.2f} seconds")


Inputs: num1 = 10.0, num2 = 5.0

Addition running | PID: 15468
Addition result: 15.0
Subtraction running | PID: 7748
Subtraction result: 5.0
Multiplication running | PID: 6300
Multiplication result: 50.0

Total execution time: 0.29 seconds


---
## 3. Key Differences

| Feature | Multi-threading | Parallel processing |
| :--- | :--- | :--- |
| **Python module** | `threading` | `multiprocessing` |
| **Execution unit** | Thread | Process |
| **Memory** | Shared within a process | Separate process memory by default |
| **Multiple CPU cores** | Limited for Python bytecode in standard CPython | Can execute CPU-bound Python code across cores |
| **Creating workers** | Generally lightweight | Generally more resource-intensive |
| **Best suited for** | I/O-bound tasks | CPU-bound tasks |
| **Communication** | Shared objects; synchronization may be needed | Often queues, pipes, or other IPC mechanisms |
| **Example** | Multiple API requests | Image processing or mathematical calculations |


---
## 4. Understand the Difference Visually

### Multi-threading (One Process, Shared Memory)
```text
┌─────────────────────────────────────────────────────────────┐
│                       One Process                           │
│                                                             │
│    [ Thread 1 ]         [ Thread 2 ]         [ Thread 3 ]   │
│         │                    │                    │         │
│         └────────────────────┼────────────────────┘         │
│                              ▼                              │
│                    ┌──────────────────┐                     │
│                    │  Shared Memory   │                     │
│                    └──────────────────┘                     │
└─────────────────────────────────────────────────────────────┘
```

---

### Parallel Processing (Multiple Processes, Separate Memory)
```text
┌────────────────────┐   ┌────────────────────┐   ┌────────────────────┐   ┌────────────────────┐
│     Process 1      │   │     Process 2      │   │     Process 3      │   │     Process 4      │
│                    │   │                    │   │                    │   │                    │
│  ┌──────────────┐  │   │  ┌──────────────┐  │   │  ┌──────────────┐  │   │  ┌──────────────┐  │
│  │   Separate   │  │   │  │   Separate   │  │   │  │   Separate   │  │   │  │   Separate   │  │
│  │    Memory    │  │   │  │    Memory    │  │   │  │    Memory    │  │   │  │    Memory    │  │
│  └──────────────┘  │   │  └──────────────┘  │   │  └──────────────┘  │   │  └──────────────┘  │
│         │          │   │         │          │   │         │          │   │         │          │
└─────────┼──────────┘   └─────────┼──────────┘   └─────────┼──────────┘   └─────────┼──────────┘
          ▼                        ▼                        ▼                        ▼
     [ CPU Core 1 ]           [ CPU Core 2 ]           [ CPU Core 3 ]           [ CPU Core 4 ]
```

**Key Takeaway:** Processes can execute simultaneously on separate CPU cores.


---
## 5. When Should You Use Each One?

Here are practical examples from **AI engineering** and **software development**:

### Use multi-threading for I/O-bound work:
- 🌐 Calling multiple LLM APIs (OpenAI, Anthropic, Gemini, local models)
- 📥 Downloading documents or media files
- 🗄️ Querying multiple databases
- 🔌 Making HTTP requests to external services

### Use multiprocessing for CPU-bound work:
- 📊 Processing large datasets
- 🧮 Performing independent mathematical calculations
- 🖼️ CPU-heavy image transformations & video filtering
- ⚙️ Running computationally intensive Python functions

---

### Real-World AI Engineering Pipeline Example: Document Processing
In a modern document-processing pipeline:
- You use **multiprocessing** for CPU-heavy tasks: converting PDF pages into high-resolution images and performing local token crunching.
- You use **multi-threading** to download source documents from cloud storage or call remote OCR and LLM APIs concurrently.

> **Rule of Thumb:** The best choice depends on whether the heavy work is performed **locally on your CPU** or **by a remote server/network service**.


---
## 6. Modern Concurrency: `concurrent.futures`

Python's standard library provides [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html) which offers a clean, unified Executor interface:
- **`ThreadPoolExecutor`:** For I/O-bound tasks.
- **`ProcessPoolExecutor`:** For CPU-bound tasks.


In [3]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch_llm_response(prompt_id):
    print(f"Sending LLM API Request {prompt_id}...")
    time.sleep(1.5)  # Simulates LLM generation latency
    return f"Response for Prompt {prompt_id}: [AI output generated]"

prompts = [1, 2, 3, 4]

start = time.perf_counter()

# Run all 4 LLM API requests concurrently
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(fetch_llm_response, prompts))

print("-" * 50)
for r in results:
    print(r)
print(f"Total time for 4 LLM requests: {time.perf_counter() - start:.2f}s (Sequential: ~6.0s)")


Sending LLM API Request 1...
Sending LLM API Request 2...
Sending LLM API Request 3...
Sending LLM API Request 4...
--------------------------------------------------
Response for Prompt 1: [AI output generated]
Response for Prompt 2: [AI output generated]
Response for Prompt 3: [AI output generated]
Response for Prompt 4: [AI output generated]
Total time for 4 LLM requests: 1.51s (Sequential: ~6.0s)


---
## 7. Summary Checklist for Students

- [x] **Multi-threading:** Multiple threads within the same process; shares memory; ideal for **I/O-bound** work (API calls, web downloads, database queries).
- [x] **Parallel Processing (Multiprocessing):** Multiple independent processes; separate memory spaces; runs across separate CPU cores; ideal for **CPU-bound** work (large calculations, image processing, machine learning).
- [x] **CPython GIL:** Restricts Python bytecode execution to one thread at a time within a single process; released during blocking I/O.
- [x] **Windows Multiprocessing Guard:** Always use `if __name__ == '__main__':` to prevent recursive process spawning on Windows and macOS.
- [x] **AI Pipelines:** Combine both techniques—multiprocessing for local data/document crunching and multi-threading for remote API / LLM inference calls.

---
**Next Class:** Asynchronous Programming in Python (`asyncio`, `async`, and `await`).
